In [1]:
from pathlib import Path
from data import ReconstructionDataset, stratified_split
from cgan import StateReconstructor
from rho import rho_from_params
import numpy as np
import qutip as qt
import h5py

root = Path.cwd().parent.parent
h5_path = f"{root}/data/train_noisy_4900.h5"
ds = ReconstructionDataset(h5_path)

In [2]:
import torch
from torch.utils.data import DataLoader
from torch import nn, optim

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
train_idx, test_idx, val_idx = stratified_split(len(ds), 67, 80, 20, 0)

train_ds = ReconstructionDataset(h5_path, indices=train_idx)
test_ds  = ReconstructionDataset(h5_path, indices=test_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [5]:
model = StateReconstructor().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss().to(device)

In [6]:
from tqdm.auto import tqdm

epochs = 50

for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    epoch_loss = 0.0

    for w, rho_true in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False):
        w = w.to(device)
        rho_true = rho_true.to(device)

        params = model(w)
        rho_hat = rho_from_params(params, N=64)
        rho_hat = rho_hat.to(device)

        loss = (
            loss_fn(rho_hat.real, rho_true.real)
            + loss_fn(rho_hat.imag, rho_true.imag)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * w.size(0)

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch + 1}: loss = {avg_loss:.6f}")

Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 1: loss = 0.005688


Epoch 2/50:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 2: loss = 0.003648


...

Epoch 48: loss = 0.000362


Epoch 49/50:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 49: loss = 0.000358


Epoch 50/50:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 50: loss = 0.000348


In [7]:
with h5py.File(h5_path, 'r') as file:
    labels = np.asarray(file['labels'])[test_idx]

In [8]:
torch.save(model.state_dict(), f"{root}/models/noisy_4900_pool4.pt")

In [9]:
model = StateReconstructor().to(device)
model.load_state_dict(torch.load(f"{root}/models/noisy_4900_pool4.pt", weights_only=True))

<All keys matched successfully>

In [10]:
LABELS = ("fock", "coherent", "vacuum", "thermal", "cat", "gkp", "binomial")
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

model.eval()
with torch.no_grad():
        
    min_fid = 1
    max_fid = 0
    fids = []
    for w, rho_true in test_loader:
        w = w.to(device)
        rho_true = rho_true.to(device)
        
        pred = model(w)
        rho_hat = rho_from_params(pred, N=64)
    
        rho_true = rho_true.detach().cpu().numpy()
        rho_hat = rho_hat.detach().cpu().numpy()
        
        fids_batch = []
        for i in range(len(rho_hat)):
            fid = qt.fidelity(qt.Qobj(rho_true[i]), qt.Qobj(rho_hat[i]))
            
            fids_batch.append(fid)
            fids.append(fid)

        mean_fids = np.mean(fids_batch)
        print(f"Średnia batcha: {mean_fids}")

        if fid > max_fid: max_fid = fid
        if fid < min_fid: min_fid = fid 
    
    print(f"Średnia całkowita: {np.mean(fids)} \n")
    print(f"Najmniejsze fid: {min_fid}")
    print(f"Największe fid: {max_fid}")

for label in LABELS:
    state_mean = np.mean([fids[i] for i in range(len(fids)) if labels[i] == LABEL_TO_ID[label]])
    print(f"Średnia stanu {label}: {state_mean}")



Średnia batcha: 0.9173778477787189
Średnia batcha: 0.9059324285495403
Średnia batcha: 0.9215924450532816
Średnia batcha: 0.9244681324042769
Średnia batcha: 0.916369388473361
Średnia batcha: 0.8916248320626597
Średnia batcha: 0.9129518244198513
Średnia batcha: 0.8983142927906402
Średnia batcha: 0.9251997316210587
Średnia batcha: 0.9888723437968892
Średnia batcha: 0.9892672944688525
Średnia batcha: 0.989312577109486
Średnia batcha: 0.9893310034847507
Średnia batcha: 0.9748578248147615
Średnia batcha: 0.9732930799241848
Średnia batcha: 0.974906735194206
Średnia batcha: 0.9724425829027488
Średnia batcha: 0.907052143058423
Średnia batcha: 0.8279615797201855
Średnia batcha: 0.8004687024577541
Średnia batcha: 0.7722485520759703
Średnia batcha: 0.8383792459609755
Średnia batcha: 0.9796377112307882
Średnia batcha: 0.9816569935511636
Średnia batcha: 0.976788830163787
Średnia batcha: 0.9811391194013669
Średnia batcha: 0.9552628934183225
Średnia batcha: 0.9125908815802055
Średnia batcha: 0.9401752